In [1]:
import pandas as pd

# Load the validated misalignment datasets from both sources.
validated_misalignments_specstory = (
    pd.read_json(
        "../misalignment-validation/validated_misalignments-specstory-chat.json"
    )
    .rename(columns={"misalignment-id": "misalignment_id"})
    .drop(columns=["repository_id_old", "session_id_old"])
)
validated_misalignments_specstory["source"] = "specstory"

validated_misalignments_swe = pd.read_json(
    "../misalignment-validation/validated_misalignments-swe-chat.json"
).rename(columns={"misalignment-id": "misalignment_id"})
validated_misalignments_swe["source"] = "swe"

# Combine both sources into one table and extract the numeric session index.
validated_misalignments = pd.concat(
    [validated_misalignments_specstory, validated_misalignments_swe], ignore_index=True
)
validated_misalignments["session_count_id"] = (
    validated_misalignments["session_id"].str.extract("session_(\\d+)").astype(int)
)

# Load the annotated misalignments and normalize the session index format.
annotated_misalignments = pd.read_json(
    "../misalignment-annotation/annotated_misalignments-all.json"
).rename(columns={"resolution_status": "resolution_status_raw"})
annotated_misalignments["session_count_id"] = (
    annotated_misalignments["session_id"].str.extract("session_(\\d+)").astype(int)
)

# Build the session mapping tables used to attach repository/session metadata.
session_mapping_specstory = pd.read_json(
    "../data/mappings/specstory_sessions_attributed.json",
    convert_dates=False,
)
session_mapping_specstory["source"] = "specstory"

session_mapping_swe = (
    pd.read_json("../data/mappings/swe_sessions_attributed.json", convert_dates=False)
    .rename(columns={"session_id": "session_sha", "created_at": "session_timestamp"})
    .drop(columns=["session_success"])
)
session_mapping_swe["source"] = "swe"

session_mapping = pd.concat(
    [session_mapping_specstory, session_mapping_swe], ignore_index=True
)

# Merge the metadata onto both datasets using the shared source/repository/session keys.
validated_misalignments = validated_misalignments.merge(
    session_mapping, how="left", on=["source", "repository_id", "session_count_id"]
)
annotated_misalignments = annotated_misalignments.merge(
    session_mapping, how="left", on=["source", "repository_id", "session_count_id"]
)

In [ ]:
# ---------- Expand multi-label fields into wide indicator columns ----------
SYMPTOM_LABELS = [f"S{i}" for i in range(1, 9)]
CAUSE_LABELS = [f"C{i}" for i in range(1, 8)]

# Symptom: boolean indicator per label, plus reasoning and other_descriptor
annotated_misalignments["symptom_reasoning"] = annotated_misalignments["symptom"].apply(
    lambda x: x.get("reasoning")
)
sym = annotated_misalignments["symptom"].apply(lambda x: x.get("labels") or [])
for s in SYMPTOM_LABELS:
    annotated_misalignments[s] = sym.apply(lambda lst: s in lst)
annotated_misalignments["symptom_other_descriptor"] = annotated_misalignments[
    "symptom"
].apply(lambda x: x.get("other_descriptor"))


# Cause: per label, store evidence_tier
# - If label not found: None
# - If label found but evidence_tier is null: "not_applicable"
# - Otherwise: the evidence_tier value
def extract_evidence_tier(lst, label):
    for d in lst:
        if d.get("label") == label:
            tier = d.get("evidence_tier")
            return "not_applicable" if tier is None else tier
    return None


annotated_misalignments["cause_reasoning"] = annotated_misalignments["cause"].apply(
    lambda x: x.get("reasoning")
)
cause_entries = annotated_misalignments["cause"].apply(lambda x: x.get("labels") or [])
for c in CAUSE_LABELS:
    annotated_misalignments[c] = cause_entries.apply(
        lambda lst, label=c: extract_evidence_tier(lst, label)
    )

# Outcome: scalar fields plus reasoning
annotated_misalignments["outcome_reasoning"] = annotated_misalignments["outcome"].apply(
    lambda x: x.get("reasoning")
)
annotated_misalignments["damage_severity"] = annotated_misalignments["outcome"].apply(
    lambda x: x.get("damage_severity")
)
annotated_misalignments["damage_locus"] = annotated_misalignments["outcome"].apply(
    lambda x: x.get("damage_locus")
)

# Resolution: scalar fields plus reasoning
annotated_misalignments["resolution_reasoning"] = annotated_misalignments[
    "resolution"
].apply(lambda x: x.get("reasoning"))
annotated_misalignments["resolution_status"] = annotated_misalignments[
    "resolution"
].apply(lambda x: x.get("status"))
annotated_misalignments["resolver"] = annotated_misalignments["resolution"].apply(
    lambda x: x.get("resolver")
)

# Drop the original multi-label fields
annotated_misalignments.drop(
    columns=[
        "count_id_original",
        "symptom",
        "cause",
        "outcome",
        "resolution",
    ],
    inplace=True,
)

In [3]:
annotated_misalignments.to_json(
    "../data/core/annotated_misalignments.json",
    orient="records",
    indent=2,
    force_ascii=False,
)

validated_misalignments.to_json(
    "../data/core/validated_misalignments.json",
    orient="records",
    indent=2,
    force_ascii=False,
)

session_mapping.to_json(
    "../data/core/session_mapping.json",
    orient="records",
    indent=2,
    force_ascii=False,
)